In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

# ===== 설정 =====
INPUT_CSV = r"C:\Users\Admin\OneDrive\바탕 화면\2차 플젝/WA_Fn-UseC_-HR-Employee-Attrition_변환.csv"  # 파일 경로
TARGET_COL = "업무평가"
ID_COLS = ["사번"]                # 절대 학습에 쓰지 않을 식별자(제거)
TOP_N = 15                        # 상위 몇 개 피처만 뽑을지

# ===== 1) 로드 =====
df = pd.read_csv(INPUT_CSV)

# ===== 2) 파생피처 생성 =====
df_feat = df.copy()
# (네가 요청했던 비율/밀도형 위주)
df_feat["프로젝트/경력"] = df_feat["참여프로젝트"] / (df_feat["경력"] + 0.5)
df_feat["근속/경력"] = df_feat["근속연차"] / (df_feat["경력"] + 0.5)
df_feat["주변평가/프로젝트"] = df_feat["주변평가"] / (df_feat["참여프로젝트"] + 0.5)
df_feat["월급/경력"] = df_feat["월급"] / (df_feat["경력"] + 0.5)
df_feat["월급/프로젝트"] = df_feat["월급"] / (df_feat["참여프로젝트"] + 0.5)
df_feat["프로젝트/교육출장"] = df_feat["참여프로젝트"] / (df_feat["전년도교육출장횟수"] + 0.5)
df_feat["경력/나이"] = df_feat["경력"] / (df_feat["나이"] + 0.5)
df_feat["월급/워라밸"] = df_feat["월급"] / (df_feat["워라밸"] + 0.5)
df_feat["초과근무율"] = df_feat["일당"] / (df_feat["근무기준시간"] + 0.5)
df_feat["현근속/총근속"] = df_feat["현회사근속년수"] / (df_feat["근속연차"] + 0.5)

# ===== 3) 학습/인코딩용 테이블 분리 =====
df_model = df_feat.drop(columns=[c for c in ID_COLS if c in df_feat.columns]).copy()
y = df_model[TARGET_COL]
X = df_model.drop(columns=[TARGET_COL])

# 범주형 라벨 인코딩(간단 버전; 원-핫 대신 라벨인코딩)
X_enc = X.copy()
for col in X_enc.select_dtypes(include=["object"]).columns:
    X_enc[col] = LabelEncoder().fit_transform(X_enc[col].astype(str))

# 타깃 인코딩
y_enc = LabelEncoder().fit_transform(y.astype(str))

# ===== 4) 중요도 기반 상위 피처 선정 =====
rf = RandomForestClassifier(
    n_estimators=500, max_depth=None, min_samples_leaf=2, random_state=42, n_jobs=-1
)
rf.fit(X_enc, y_enc)
imp = pd.Series(rf.feature_importances_, index=X_enc.columns).sort_values(ascending=False)
top_features = imp.head(TOP_N).index.tolist()

print("[선정된 상위 피처]", *top_features, sep="\n- ")

# ===== 5) 그 피처만 추출 + 타깃, 저장 =====
selected_df = df_feat[top_features + [TARGET_COL]].copy()
OUTPUT_CSV = INPUT_CSV.rsplit(".",1)[0] + f"_selected_top{TOP_N}.csv"
selected_df.to_csv(OUTPUT_CSV, index=False)
print(f"\n저장 완료 → {OUTPUT_CSV}")


[선정된 상위 피처]
- 월급/경력
- 시급
- 월급/프로젝트
- 월급
- 월급/워라밸
- 일당
- 초과근무율
- 경력/나이
- 나이
- 거리
- 프로젝트/경력
- 현근속/총근속
- 근속/경력
- 프로젝트/교육출장
- 현회사근속년수

저장 완료 → C:\Users\Admin\OneDrive\바탕 화면\2차 플젝/WA_Fn-UseC_-HR-Employee-Attrition_변환_selected_top15.csv
